In [ ]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import OrdinalEncoder, RobustScaler
from matplotlib import pyplot as plt
from seaborn import heatmap, pairplot
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix

from google.colab import drive
drive.mount('/content/drive')

: 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
data = pd.read_csv("/content/drive/MyDrive/CSE422 Project/nasa.csv")
# for name, dtype in data.dtypes.iteritems():
#     print(name, dtype)
data





There are 40 features including the target feature 'Hazardous' and 4687 data points in the dataset.



# Feature Engineering:


## Removing Irrelevant Columns:

In [ ]:
data.columns

The features 'Neo Reference ID', 'Name', 'Close Approach Date', 'Epoch Date Close Approach', 'Orbit ID' and 'Orbit Determination Date' are insignificant in training our ML model.



In [ ]:
data["Orbit ID"]

In [ ]:
data.drop(['Neo Reference ID', 'Name', 'Close Approach Date', 'Epoch Date Close Approach', 'Orbit ID', 'Orbit Determination Date'], axis = 1, inplace = True)
data.columns

In [ ]:
data["Orbiting Body"]

In [ ]:
data["Equinox"]

In [ ]:
data['Orbiting Body'].unique(), data['Equinox'].unique()

As the features 'Orbiting Body' and 'Equinox' holds only one unique value, they are not important as well.



In [ ]:
data.drop(['Orbiting Body', 'Equinox'], axis = 1, inplace = True)
data.columns

## Handling Missing Values:


In [ ]:
data.isnull().sum()


In [ ]:
# data.fillna(replacement_values, inplace=True)#if there was any null value we would have handled it by replacing the null values through a value

There are no missing values in the dataset.



## Handling Duplicate Values:

In [ ]:
data.duplicated().sum()

The dataset has no duplicate values.



In [ ]:
#jodi duplicate values thakto then ebhabe handle kortam
#data.duplicated()  # Returns a boolean Series indicating duplicate rows
# data[data.duplicated()]
# data.drop_duplicates(inplace=True)

## Feature Encoding

In [ ]:
data.dtypes

Among all the features, only our target feature 'Hazardous' is categorical in nature.


In [ ]:
#we later used label encoder after we realized that we don't really need to show any hierarchy for the intensity of the hazards of the asteroids
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
data['Hazardous'] = label_encoder.fit_transform(data['Hazardous'])
print(data['Hazardous'].head(5))

In [ ]:
# data['Hazardous'] = OrdinalEncoder(dtype = np.int64).fit_transform(data[['Hazardous']])
# data.Hazardous.head(5)

## Feature Selection:

In [ ]:
plt.figure(figsize = (40,40))
heatmap(data.corr(), annot = True)
#Redundancy
#Multicollinearity
#Improved Model Performance:less prone to overfitting and can be more interpretable
#Computational Efficiency

One between the pair of features with correlation value |x| >= .9 are to be eliminated.



In [ ]:
data.drop(['Est Dia in KM(max)', 'Est Dia in M(min)', 'Est Dia in M(max)', 'Est Dia in Miles(min)', 'Est Dia in Miles(max)', 'Est Dia in Feet(min)', 'Est Dia in Feet(max)', 'Relative Velocity km per hr', 'Miles per hour', 'Miss Dist.(lunar)', 'Miss Dist.(kilometers)', 'Miss Dist.(miles)', 'Mean Motion', 'Perihelion Time', 'Orbital Period', 'Aphelion Dist', 'Semi Major Axis'], axis = 1, inplace = True)
data.columns

In [ ]:
plt.figure(figsize = (40,40))
heatmap(data.corr(), annot = True)

## Feature Scaling:

In [ ]:
plt.figure(figsize = (20,5))
data.boxplot(column = ['Absolute Magnitude', 'Relative Velocity km per sec', 'Orbit Uncertainity', 'Jupiter Tisserand Invariant', 'Inclination'])
#jupiter tisserand invariant: characterize the type of orbit a small object (such as a comet or asteroid)
#  has around the Sun in relation to Jupiter's influence. It's named after the French astronomer Félix Tisserand

In [ ]:
plt.figure(figsize = (20,5))
data.boxplot(column = ['Est Dia in KM(min)', 'Miss Dist.(Astronomical)', 'Minimum Orbit Intersection', 'Eccentricity', 'Perihelion Distance'])

In [ ]:
plt.figure(figsize = (5,5))
data.boxplot(column = ['Epoch Osculation'])

In [ ]:
# epoch osculation is a method used to define the orbital elements of a celestial object at a particular reference time or epoch.
# Orbital elements describe the shape, orientation, and size of an object's orbit.

In [ ]:
plt.figure(figsize = (5,5))
data.boxplot(column = ['Asc Node Longitude', 'Perihelion Arg', 'Mean Anomaly'])




As the dataset is prone to outliers, it should be scaled using a scaler which is not sensitive to outliers .



In [ ]:
data[data.columns[:len(data.columns)-1]] = RobustScaler().fit_transform(data[data.columns[:len(data.columns)-1]])
data.head(5)

## Data Analysis

As the Target Feature is categorical in nature, Classifier ML Models are to be used.



In [ ]:
pairplot(data, hue='Hazardous', corner = True)

As the dataset is not linearly separable, Non-linear Classifier ML Models are to be used.



In [ ]:
plt.figure(figsize = (30,10))
data.boxplot(column = ['Absolute Magnitude', 'Miss Dist.(Astronomical)', 'Orbit Uncertainity', 'Minimum Orbit Intersection', 'Jupiter Tisserand Invariant', 'Eccentricity', 'Asc Node Longitude', 'Perihelion Distance', 'Perihelion Arg', 'Mean Anomaly'])

In [ ]:
plt.figure(figsize = (10,5))
data.boxplot(column = ['Est Dia in KM(min)','Relative Velocity km per sec', 'Inclination'])

In [ ]:
plt.figure(figsize = (10,5))
data.boxplot(column = ['Epoch Osculation'])

The features with significant number of outliers are to be removed.






In [ ]:
data.drop(['Est Dia in KM(min)','Relative Velocity km per sec', 'Epoch Osculation', 'Inclination'], axis = 1, inplace = True)
data.columns

# Model Application

## Splitting Dataset

In [ ]:
features = data[data.columns[:len(data.columns)-1]]
target = data[data.columns[-1]]

The features would be splited with a Train-Test ratio of 70:30.



In [ ]:
trainFeatures, testFeatures, trainTarget, testTarget = train_test_split(features, target, test_size = 0.3, random_state = 1)

## Random Forest Classifier


In [ ]:
acc_val, f1_val, precision_val, recall_val = [],[],[],[]

for i in range(1,50,2):
    model = RandomForestClassifier(n_estimators = i,criterion = 'gini', random_state = 1)
    model.fit(trainFeatures, trainTarget)
    predTarget = model.predict(testFeatures)
    acc_val.append(round(accuracy_score(testTarget, predTarget)*100, 2))
    f1_val.append(round(f1_score(testTarget, predTarget)*100, 2))
    precision_val.append(round(precision_score(testTarget, predTarget)*100, 2))
    recall_val.append(round(recall_score(testTarget, predTarget)*100, 2))

plt.plot(range(1, 50, 2), acc_val, label='Accuracy') #BLUE
plt.plot(range(1, 50, 2), f1_val, label='F1 Score') #ORANGE
plt.plot(range(1, 50, 2), precision_val, label='Precision')#Green
plt.plot(range(1, 50, 2), recall_val, label='Recall')#RED
plt.show()

From the graph it is clear that Random Forest Classifier produces best output on this dataset when n_estimators = 7.



### Result Analysis for RFC:



In [ ]:
model = RandomForestClassifier(n_estimators = 7, criterion = 'gini', random_state = 1)
model.fit(trainFeatures, trainTarget)
predTarget = model.predict(testFeatures)

accuracy = round(accuracy_score(testTarget, predTarget) * 100, 2)
f1 = round(f1_score(testTarget, predTarget) * 100, 2)
precision = round(precision_score(testTarget, predTarget) * 100, 2)
recall = round(recall_score(testTarget, predTarget) * 100, 2)

print(f'Model: Random Forest\n'
      f'Accuracy Score: {accuracy}%\n'
      f'F1 Score: {f1}%\n'
      f'Precision Score: {precision}%\n'
      f'Recall Score: {recall}%')
heatmap(confusion_matrix(testTarget, predTarget), annot=True)

## Support Vector Classifier:

In [ ]:
acc_val, f1_val, precision_val, recall_val = [],[],[],[]

for i in range(1,50,2):
    model = SVC(C = i, kernel = 'rbf')
    model.fit(trainFeatures, trainTarget)
    predTarget = model.predict(testFeatures)
    acc_val.append(round(accuracy_score(testTarget, predTarget)*100, 2))
    f1_val.append(round(f1_score(testTarget, predTarget)*100, 2))
    precision_val.append(round(precision_score(testTarget, predTarget)*100, 2))
    recall_val.append(round(recall_score(testTarget, predTarget)*100, 2))

plt.plot(range(1, 50, 2), acc_val, label='Accuracy') #BLUE
plt.plot(range(1, 50, 2), f1_val, label='F1 Score') #ORANGE
plt.plot(range(1, 50, 2), precision_val, label='Precision')#Green
plt.plot(range(1, 50, 2), recall_val, label='Recall')#RED
plt.show()

From the graph it is clear that Support Vector Classifier produces best output on this dataset when C = 15.



### Result Analysis for SVC:



In [ ]:
model = SVC(C = 15, kernel = 'rbf')
model.fit(trainFeatures, trainTarget)
predTarget = model.predict(testFeatures)

accuracy = round(accuracy_score(testTarget, predTarget) * 100, 2)
f1 = round(f1_score(testTarget, predTarget) * 100, 2)
precision = round(precision_score(testTarget, predTarget) * 100, 2)
recall = round(recall_score(testTarget, predTarget) * 100, 2)

print(f'Model: Random Forest\n'
      f'Accuracy Score: {accuracy}%\n'
      f'F1 Score: {f1}%\n'
      f'Precision Score: {precision}%\n'
      f'Recall Score: {recall}%')

heatmap(confusion_matrix(testTarget, predTarget), annot=True)

## K-Nearest Neighbor Classifier

In [ ]:
acc_val, f1_val, precision_val, recall_val = [],[],[],[]

for i in range(1,50,2):
    model = KNeighborsClassifier(n_neighbors = i, weights = 'distance')
    model.fit(trainFeatures, trainTarget)
    predTarget = model.predict(testFeatures)
    acc_val.append(round(accuracy_score(testTarget, predTarget)*100, 2))
    f1_val.append(round(f1_score(testTarget, predTarget)*100, 2))
    precision_val.append(round(precision_score(testTarget, predTarget)*100, 2))
    recall_val.append(round(recall_score(testTarget, predTarget)*100, 2))

plt.plot(range(1, 50, 2), acc_val, label='Accuracy') #BLUE
plt.plot(range(1, 50, 2), f1_val, label='F1 Score') #ORANGE
plt.plot(range(1, 50, 2), precision_val, label='Precision')#Green
plt.plot(range(1, 50, 2), recall_val, label='Recall')#RED
plt.show()

From the graph it is clear that K-Nearest Neighbor Classifier produces best output on this dataset when n_neighbors = 48.



### Result Analysis for KNC:



In [ ]:
model = KNeighborsClassifier(n_neighbors = 48, weights = 'distance')
model.fit(trainFeatures, trainTarget)
predTarget = model.predict(testFeatures)

accuracy = round(accuracy_score(testTarget, predTarget) * 100, 2)
f1 = round(f1_score(testTarget, predTarget) * 100, 2)
precision = round(precision_score(testTarget, predTarget) * 100, 2)
recall = round(recall_score(testTarget, predTarget) * 100, 2)

print(f'Model: Random Forest\n'
      f'Accuracy Score: {accuracy}%\n'
      f'F1 Score: {f1}%\n'
      f'Precision Score: {precision}%\n'
      f'Recall Score: {recall}%')

heatmap(confusion_matrix(testTarget, predTarget), annot=True)

Among the three Classifiers used on the dataset, Random Forest Classifier has the highest value of accuracy score,f1 score, precision score, recall score and the lowest value of false positives and false negetives. Thereby it is concluded that for this dataset, Random Forest Classifier produced the best outcome.